<a href="https://colab.research.google.com/github/Keistkmiya/Tugas1-MachineLearning/blob/main/Tugas1_Chapter6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 6: Algorithm Chains and Pipelines

## Setup and Library Imports

In [1]:
!pip install mglearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.4/581.4 kB 6.4 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mglearn
import warnings
from sklearn.svm import SVC
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

%matplotlib inline

## Building Pipelines
Dalam machine learning, Kita sering kali harus melakukan prapemrosesan data (seperti scaling) sebelum melatih model. Jika kita memisahkan proses scaling dan cross-validation, kita bisa tidak sengaja membocorkan informasi dari data uji ke data latih (data leakage). Untuk mencegahnya, kita menggunakan kelas `Pipeline` dari scikit-learn.

In [3]:
cancer = load_breast_cancer()
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, random_state=0)

scaler = MinMaxScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)

svm = SVC()
svm.fit(X_train_scaled, y_train)
X_test_scaled = scaler.transform(X_test)

print("Test score: {:.2f}".format(svm.score(X_test_scaled, y_test)))

Test score: 0.97


### Using the Pipeline Class
Daripada melakukan scaling dan melatih model secara manual dan terpisah seperti di atas, Kita bisa menggabungkannya ke dalam satu objek Pipeline. Ini membuat alur kerja menjadi jauh lebih rapi, otomatis, dan aman dari kebocoran data saat dievaluasi.

In [4]:
from sklearn.pipeline import Pipeline

pipe = Pipeline([("scaler", MinMaxScaler()), ("svm", SVC())])

pipe.fit(X_train, y_train)

print("Test score with Pipeline: {:.2f}".format(pipe.score(X_test, y_test)))

Test score with Pipeline: 0.97


### Using Pipelines in Grid Searches

Menggabungkan Pipeline dengan Grid Search adalah cara terbaik untuk memastikan proses evaluasi kita benar-benar aman dari kebocoran data (data leakage). Saat kita melakukan cross-validation di dalam Grid Search, scaler hanya akan dilatih menggunakan data pada *fold* latih, dan tidak pernah menyentuh data pada *fold* validasi.

Perlu diperhatikan, untuk mengatur parameter model di dalam pipeline, kita harus menuliskan nama langkahnya (misalnya `svm`) diikuti dengan dua garis bawah (`__`), barulah nama parameternya.

In [5]:
from sklearn.model_selection import GridSearchCV

param_grid = {'svm__C': [0.001, 0.01, 0.1, 1, 10, 100],
              'svm__gamma': [0.001, 0.01, 0.1, 1, 10, 100]}

pipe = Pipeline([("scaler", MinMaxScaler()), ("svm", SVC())])

grid = GridSearchCV(pipe, param_grid=param_grid, cv=5)
grid.fit(X_train, y_train)

print("Best cross-validation accuracy: {:.2f}".format(grid.best_score_))
print("Test set score: {:.2f}".format(grid.score(X_test, y_test)))
print("Best parameters: {}".format(grid.best_params_))

Best cross-validation accuracy: 0.98
Test set score: 0.97
Best parameters: {'svm__C': 1, 'svm__gamma': 1}


### Convenient Pipeline Creation with `make_pipeline`

Menuliskan nama untuk setiap langkah (seperti "scaler" atau "svm") secara manual terkadang merepotkan, apalagi jika pipeline kita sangat panjang. Untuk menyederhanakannya, scikit-learn menyediakan fungsi `make_pipeline` yang akan secara otomatis memberikan nama standar untuk setiap langkah berdasarkan nama kelas algoritmanya.

In [6]:
from sklearn.pipeline import make_pipeline

pipe_short = make_pipeline(MinMaxScaler(), SVC(C=100))

print("Pipeline steps:\n{}".format(pipe_short.steps))

pipe_short.fit(X_train, y_train)
print("Score with make_pipeline: {:.2f}".format(pipe_short.score(X_test, y_test)))

Pipeline steps:
[('minmaxscaler', MinMaxScaler()), ('svc', SVC(C=100))]
Score with make_pipeline: 0.97


### Accessing Step Attributes

Terkadang kita perlu melihat lebih dalam ke model yang sudah dilatih di dalam pipeline, misalnya untuk melihat bobot (koefisien) dari sebuah model regresi. Karena model tersebut "dibungkus" oleh pipeline, kita tidak bisa memanggilnya secara langsung. Kita harus menggunakan atribut `named_steps` untuk memanggil nama langkahnya secara spesifik.

In [7]:
from sklearn.linear_model import LogisticRegression

pipe = make_pipeline(MinMaxScaler(), LogisticRegression(max_iter=1000))
pipe.fit(X_train, y_train)

print("Langkah-langkah di dalam pipeline:\n{}".format(pipe.named_steps))
print("\nKoefisien Logistic Regression:\n{}".format(
    pipe.named_steps["logisticregression"].coef_))

Langkah-langkah di dalam pipeline:
{'minmaxscaler': MinMaxScaler(), 'logisticregression': LogisticRegression(max_iter=1000)}

Koefisien Logistic Regression:
[[-1.69572541 -1.64594793 -1.67124184 -1.43804269 -0.81241207 -0.55359388
  -1.22338425 -1.95917219 -0.80462433  0.78458703 -1.22838321 -0.04772064
  -0.9802163  -0.7944732   0.29203657  0.67251348  0.3215652  -0.20003092
   0.19642781  0.59359988 -2.15393219 -1.83395061 -1.99575797 -1.55487793
  -1.24354458 -0.85315882 -1.21600084 -2.54947627 -1.25370237 -0.38466246]]


### Grid-Searching Preprocessing Steps and Model Parameters

Kekuatan paling hebat dari penggabungan Pipeline dan Grid Search adalah kita bisa mencari parameter terbaik tidak hanya untuk model akhir (seperti SVM), tetapi juga parameter untuk tahap prapemrosesan datanya secara bersamaan!

Sebagai contoh, kita bisa mencari tahu berapa banyak komponen PCA (Principal Component Analysis) yang paling optimal jika digabungkan dengan kombinasi nilai C dan gamma tertentu pada SVM.

In [8]:
from sklearn.decomposition import PCA

pipe = Pipeline([("scaler", MinMaxScaler()), ("pca", PCA()), ("svm", SVC())])

param_grid = {'pca__n_components': [1, 2, 3],
              'svm__C': [0.001, 0.01, 0.1, 1, 10, 100],
              'svm__gamma': [0.001, 0.01, 0.1, 1, 10, 100]}

grid = GridSearchCV(pipe, param_grid=param_grid, cv=5)
grid.fit(X_train, y_train)

print("Akurasi cross-validation terbaik: {:.2f}".format(grid.best_score_))
print("Akurasi test set: {:.2f}".format(grid.score(X_test, y_test)))
print("Parameter terbaik:\n{}".format(grid.best_params_))

Akurasi cross-validation terbaik: 0.96
Akurasi test set: 0.97
Parameter terbaik:
{'pca__n_components': 3, 'svm__C': 100, 'svm__gamma': 0.01}
